<a href="https://colab.research.google.com/github/ReneeKang/pandas_pyspark_pycaret/blob/main/Spark_join_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

강의 상황별 최적의 Spark Join 선택 가이드

https://www.youtube.com/watch?v=kSJ8HTumwVc&list=PL918La0-0ENbTKGm_2SOIsmpVIDcFgx-t&index=4

# 1. JOIN 최적화의 필요성

#### 1) JOIN은 노드간의 데이터 이동(Shuffle)이 발생하는 연산으로 리소스 소모 비용이 큰 연산 중 하나입니다.

#### 2) 잘못된 전략을 사용하면 셔플(Shuffle)이 과도하게 발생해 실행 시간이 길어지거나, 메모리 부족(OOM)으로 작업이 실패할 수 있습니다.

#### 3) Spark는 기본적으로 "빠른 것보다 안전한 것"을 선택하지만, 데이터 특성에 맞춰 최적의 전략을 선택하면 성능을 크게 향상시킬 수 있습니다.

### 알아볼 JOIN 방법
- Sort Merge Join
- Shuffle Hash Join
- Broadcast Join

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
import random
from datetime import datetime, timedelta

# SparkSession 생성
spark = (
    SparkSession.builder
    .appName("JoinStrategyDemo")
    .getOrCreate()
)

# 1. 차원 테이블 (카테고리) 생성
num_categories = 100  # 100개 미만
categories_data = [(i, f"Category_{i}") for i in range(num_categories)]
dim_category_schema = T.StructType([
    T.StructField("id", T.IntegerType(), False),
    T.StructField("category_name", T.StringType(), False)
])
dim_category = spark.createDataFrame(categories_data, dim_category_schema)

# 2. 팩트 테이블 (판매) 생성
num_sales = 1000000 # 100만개
def generate_sales_data(n):
    data = []
    base_date = datetime(2024, 1, 1)
    for i in range(n):
        product_id = f"PROD_{random.randint(1, 50000)}"
        category_id = random.randint(0, num_categories - 1)
        amount = round(random.uniform(10.0, 5000.0), 2)
        sale_date = base_date + timedelta(days=random.randint(0, 365))
        data.append((i, product_id, category_id, amount, sale_date.strftime("%Y-%m-%d")))
    return data

sales_data = generate_sales_data(num_sales)

fact_sales_schema = T.StructType([
    T.StructField("sale_id", T.IntegerType(), False),
    T.StructField("product_id", T.StringType(), False),
    T.StructField("category_id", T.IntegerType(), False),
    T.StructField("amount", T.DoubleType(), False),
    T.StructField("sale_date", T.StringType(), False)
])
fact_sales = spark.createDataFrame(sales_data, fact_sales_schema)


print(f"Dimension table 'dim_category' created with {dim_category.count()} rows.")
print(f"Fact table 'fact_sales' created with {fact_sales.count()} rows.")

Dimension table 'dim_category' created with 100 rows.
Fact table 'fact_sales' created with 1000000 rows.


In [2]:
dim_category.printSchema()
fact_sales.printSchema()

root
 |-- id: integer (nullable = false)
 |-- category_name: string (nullable = false)

root
 |-- sale_id: integer (nullable = false)
 |-- product_id: string (nullable = false)
 |-- category_id: integer (nullable = false)
 |-- amount: double (nullable = false)
 |-- sale_date: string (nullable = false)



In [3]:
dim_category.show()
fact_sales.show()

+---+-------------+
| id|category_name|
+---+-------------+
|  0|   Category_0|
|  1|   Category_1|
|  2|   Category_2|
|  3|   Category_3|
|  4|   Category_4|
|  5|   Category_5|
|  6|   Category_6|
|  7|   Category_7|
|  8|   Category_8|
|  9|   Category_9|
| 10|  Category_10|
| 11|  Category_11|
| 12|  Category_12|
| 13|  Category_13|
| 14|  Category_14|
| 15|  Category_15|
| 16|  Category_16|
| 17|  Category_17|
| 18|  Category_18|
| 19|  Category_19|
+---+-------------+
only showing top 20 rows
+-------+----------+-----------+-------+----------+
|sale_id|product_id|category_id| amount| sale_date|
+-------+----------+-----------+-------+----------+
|      0|PROD_29002|         20|3075.97|2024-08-15|
|      1|PROD_39994|         82|1331.35|2024-11-25|
|      2|PROD_46822|         46|3811.57|2024-10-29|
|      3|PROD_14013|          0|2327.78|2024-02-28|
|      4|PROD_11901|          6|1235.86|2024-09-13|
|      5|PROD_27822|         79| 419.22|2024-12-31|
|      6|PROD_19969|       

# 1. Sort Merge Join

![](https://velog.velcdn.com/images/newnew_daddy/post/5da1c7a1-ff61-4673-9cb1-3aa43b1c971b/image.png)

### **핵심 개념**
- **"크든 작든 안전하게 가는 정석 루트 (Spark 기본값)"**
- 두 테이블을 조인 키 기준으로 **Shuffle**한 뒤, 각 파티션에서 **Sort(정렬)**하고 순차적으로 **Merge**합니다.
- 정렬된 데이터를 순차 탐색하므로 메모리에 전체를 올릴 필요가 없고, 메모리 부족 시 **Spill(디스크 쓰기)**이 가능해 가장 안전합니다.

### **비유**
> "두 명단을 번호순으로 정렬해 놓고, 손가락 두 개로 같이 훑어내려가며 비교하기"

### **언제 사용하는가?**
- **양쪽 테이블이 모두 클 때**
- 메모리 제한이 엄격하거나 데이터 스큐가 있어 OOM 위험이 있을 때

In [4]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
spark.conf.set("spark.sql.join.preferSortMergeJoin", True)

In [5]:
smj_df = (
    fact_sales
    .join(
        dim_category,
        fact_sales.category_id == dim_category.id,
        "inner"
    )
)

smj_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [category_id#4], [id#0], Inner
   :- Sort [category_id#4 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(category_id#4, 200), ENSURE_REQUIREMENTS, [plan_id=105]
   :     +- Scan ExistingRDD[sale_id#2,product_id#3,category_id#4,amount#5,sale_date#6]
   +- Sort [id#0 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(id#0, 200), ENSURE_REQUIREMENTS, [plan_id=106]
         +- Scan ExistingRDD[id#0,category_name#1]




#### 주요 단계 분석

- `Scan`
  - 원본 데이터(RDD, Parquet 등)를 읽어옵니다.
- `Exchange (hashpartitioning)`
  - Shuffle 단계입니다. 조인 키를 기준으로 데이터를 재배치합니다.
  - 동일한 조인 키를 가진 데이터들을 물리적으로 같은 파티션(노드)으로 모으는 과정입니다.
- `Sort`
  - 각 파티션 내에서 데이터를 조인 키 순서대로 정렬합니다.
  - 이 과정이 끝나야만 두 데이터를 순차적으로 읽으며 비교하는 'Merge'가 가능해집니다.

```
SortMergeJoin
  :- [Left 테이블 준비]: Scan(읽기) -> Exchange(모으기) -> Sort(정렬)
  +- [Right 테이블 준비]: Scan(읽기) -> Exchange(모으기) -> Sort(정렬)
```

# 2. Shuffle Hash Join

![](https://velog.velcdn.com/images/newnew_daddy/post/6d811dff-4b10-40e0-8285-0532ebe9a439/image.png)

### **핵심 개념**
- **"정렬(Sort) 없이 해시맵(HashMap)으로 빠르게"**
- 두 테이블을 조인 키 기준으로 **Shuffle**한 뒤, 작은 쪽 파티션 데이터를 메모리에 올려 **해시맵**을 만듭니다.
- 정렬 과정이 없어 빠르지만, 해시맵을 메모리에 올려야 하므로 데이터 스큐(Skew)가 있거나 메모리가 부족하면 **OOM**이 발생하기 쉽습니다.

### **비유**
> "지역별로 모아놓고(Shuffle), 작은 명단만 책상 위에 펼쳐서(Hash Map) 큰 명단 보며 바로 찾기"

### **언제 사용하는가?**
- **키 분포가 고르고**, 한 파티션 데이터가 메모리에 충분히 들어갈 때

In [6]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # broadcast 금지
spark.conf.set("spark.sql.join.preferSortMergeJoin", False)  # sort merge 비선호
spark.conf.set("spark.sql.join.enableShuffleHashJoin", True)

In [7]:
shj_df = (
    fact_sales
    .join(
        dim_category.hint("SHUFFLE_HASH"),
        fact_sales.category_id == dim_category.id,
        "inner"
    )
)
shj_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- ShuffledHashJoin [category_id#4], [id#0], Inner, BuildRight
   :- Exchange hashpartitioning(category_id#4, 200), ENSURE_REQUIREMENTS, [plan_id=123]
   :  +- Scan ExistingRDD[sale_id#2,product_id#3,category_id#4,amount#5,sale_date#6]
   +- Exchange hashpartitioning(id#0, 200), ENSURE_REQUIREMENTS, [plan_id=124]
      +- Scan ExistingRDD[id#0,category_name#1]




#### 주요 단계 분석
*   **`Exchange (hashpartitioning)`**:
    - **Shuffle** 단계입니다. Sort Merge Join과 마찬가지로 조인 키를 기준으로 데이터를 재배치합니다.
    - 양쪽 테이블의 데이터를 동일한 해시 파티션으로 모아 "서로 만날 수 있는 상태"로 만듭니다.
*   **`BuildRight` (중요!)**:
    - 조인에 참여하는 테이블 중 **오른쪽(Right) 테이블**을 사용하여 메모리에 **해시 맵(Hash Map)**을 생성한다는 의미입니다.
    - 보통 더 작은 테이블이 'Build' 대상으로 선택됩니다.
*   **`ShuffledHashJoin`**:
    - Sort Merge Join과 달리 **`Sort` 단계가 없습니다.**
    - 각 파티션 내에서 한쪽(Build Side) 데이터를 해시 맵에 다 집어넣고, 반대편(Stream Side) 데이터를 한 줄씩 읽으며 해시 맵에서 매칭되는 값을 즉시 찾습니다.

#### Sort Merge Join과의 차이점
| 특징 | Sort Merge Join | Shuffled Hash Join |
| :--- | :--- | :--- |
| **사전 작업** | Shuffle + **Sort** | Shuffle만 수행 (정렬 X) |
| **동작 방식** | 두 포인터를 정렬된 순서로 이동 | 한쪽을 해시 맵으로 만들어 조회 |
| **메모리** | 메모리 부족 시 디스크 활용 가능 (안전) | 해시 맵이 한 파티션 메모리보다 크면 **OOM 발생** |

#### 📋 요약
Shuffled Hash Join은 **"데이터를 모으기는 하되(Shuffle), 정렬 비용을 아끼고 해시 맵의 빠른 검색(O(1))을 활용하겠다"**는 전략입니다.
양쪽 다 셔플이 발생하므로 네트워크 비용은 동일하지만, 정렬(Sort) 단계가 생략되어 CPU 사용량을 줄일 수 있는 장점이 있습니다.

# 3. Broadcast Join

![](https://velog.velcdn.com/images/newnew_daddy/post/0b7087b2-88ad-4e90-a868-8b942a59c165/image.png)

### **핵심 개념**
- **"작은 테이블을 모든 노드에 복사해서 셔플 제거"**
- 작은 테이블(Dimension)을 Driver가 모든 Executor에 **복사(Broadcast)**합니다.
- **셔플(Network I/O)이 전혀 발생하지 않아** 가장 빠릅니다. 단, 테이블이 너무 크면 Driver 메모리가 부족하거나 네트워크 부하가 걸립니다.

### **언제 사용하는가?**
- **한쪽 테이블이 아주 작을 때** (기본 10MB 이하)
- 차원 테이블(코드, 매핑 테이블)과 조인할 때

In [8]:
from pyspark.sql.functions import broadcast

bj_df = (
    fact_sales
    .join(
        broadcast(dim_category),
        fact_sales.category_id == dim_category.id,
        "inner"
    )
)

bj_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [category_id#4], [id#0], Inner, BuildRight, false
   :- Scan ExistingRDD[sale_id#2,product_id#3,category_id#4,amount#5,sale_date#6]
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=138]
      +- Scan ExistingRDD[id#0,category_name#1]




#### 주요 단계 분석
*   **`BroadcastExchange` (가장 핵심)**:
    - 데이터 전송 방식이 Shuffle(Exchange)과 완전히 다릅니다.
    - **Driver**가 작은 테이블(`dim_category`)의 데이터를 모두 수집한 뒤, 모든 **Executor**들에게 통째로 복사본을 보냅니다.
    - 이 과정 덕분에 큰 테이블인 `fact_sales`(Left Side)는 데이터를 옮길 필요 없이 자기 자리(Local)에서 바로 읽기만 하면 됩니다.
*   **`BuildRight`**:
    - 방송된(Broadcasted) 오른쪽 테이블을 메모리에 해쉬 맵(Hash Map)으로 구현했다는 의미입니다.
*   **`BroadcastHashJoin`**:
    - 큰 테이블의 데이터를 한 줄씩 읽으면서(Streaming), 메모리에 올라와 있는 작은 테이블의 해쉬 맵에서 즉시 매칭되는 값을 찾습니다.

| 비교 대상 | Shuffle 기반 Join (SMJ, SHJ) | Broadcast Join |
| :--- | :--- | :--- |
| **네트워크 부하** | 양쪽 테이블 모두 대량 이동 (전체 데이터 이동) | 작은 테이블만 복사해서 전송 (최소화) |
| **디스크 I/O** | Shuffle Write/Read 발생 (디스크 쓰기 발생 가능) | Shuffle 자체가 없으므로 디스크 오버헤드 거의 없음 |
| **정렬/해시** | 정렬(Sort) 또는 전체 셔플 해시 필요 | 작은 쪽만 해시 맵으로 빌드 |

In [9]:
print(spark.conf.get("spark.sql.adaptive.enabled"))
print(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

true
-1


# 4. 세 가지 방식 비교 및 요약

| 구분 | Broadcast Join | Shuffle Hash Join | Sort Merge Join |
|------|-------------------|-------------------|-----------------|
| **셔플 여부** | ❌ (복사) | ✅ | ✅ |
| **정렬 여부** | ❌ | ❌ | ✅ |
| **장점** | **가장 빠름** (No Shuffle) | 정렬 오버헤드 없음 | **가장 안전함** (Spill 가능) |
| **단점** | 메모리 많이 씀 (Driver/Executor) | OOM 위험 (Skew 취약) | 정렬 오버헤드 존재 |
| **추천 상황** | **아주 작은 테이블** (코드성) | **중간 크기** & 분포 고름 | **대용량** & 안전성 우선 |

### **한 줄 요약 가이드**
1. **아주 작다** → `Broadcast Join` (자동 또는 힌트)
2. **조금 작다 & 분포가 좋다** → `Shuffle Hash Join`
3. **둘 다 크다** → `Sort Merge Join` (기본값, 속 편함)